In [12]:
from datetime import date

import pandas as pd
from sqlalchemy import func, or_

from epo.tipdata.patstat import PatstatClient
from epo.tipdata.patstat.database.models import (
    TLS201_APPLN,               # applications
    TLS202_APPLN_TITLE,         # titles
    TLS203_APPLN_ABSTR,         # abstracts
    TLS206_PERSON,              # persons (applicants and inventors)
    TLS207_PERS_APPLN,          # person <-> application links
    TLS209_APPLN_IPC,           # IPC classification
    TLS224_APPLN_CPC,           # CPC classification
    TLS231_INPADOC_LEGAL_EVENT, # legal events
)

In [13]:
patstat = PatstatClient()          # default: env="TEST"; use PatstatClient(env="PROD") for the full database
db = patstat.orm()                 # SQLAlchemy session used by all ORM queries below

This client instance is currently configured to use a test dataset with reduced number of publications (~10K).
Use PatstatClient(env='PROD') to use the complete PATSTAT dataset (>140M publications).
Use PatstatClient(env='TEST') to use the test dataset and avoid displaying this warning



In [14]:
DRONE_KEYWORDS = (
    # General UAV terminology
    "unmanned aerial vehicle",
    "unmanned aerial system",
    "unmanned aircraft",
    "uncrewed aerial vehicle",
    "uncrewed aircraft",
    "unmanned air vehicle",
    "remotely piloted aircraft",
    "remotely piloted aerial vehicle",
    "remotely controlled aircraft",
    "unmanned flying vehicle",

    # Common acronyms
    "UAV",
    "RPAS",

    # Drone types
    "quadcopter",
    "quadrotor",
    "quad-rotor",
    "hexacopter",
    "hexarotor",
    "octocopter",

    # Additional UAV terminology
    "aerial drone",
    "flying drone",
    "drone aircraft",
    "fixed-wing drone",
    "flying robot",
    "micro air vehicle",
)

DRONE_KEYWORDS = (
    # "unmanned/unpiloted" + aircraft/aerial (both papers)
    "unmanned aerial", "unmanned aircraft", "unmanned air vehicle",
    "unpiloted aerial", "unpiloted aircraft", "unpiloted air vehicle",
    # remote* within 1 word of aircraft (DTI)
    "remotely piloted aircraft", "remote controlled aircraft", "remote-controlled aircraft",
    "micro air vehicle",                                    # DTI
    # rotor types (Gorry)
    "quadrotor", "quadrocopter", "quadcopter", "multicopter", "multirotor", "multi-rotor",
    "hexacopter", "octocopter",
    " uav", "uav ",
    "drone",                                                # contested: keep only if hand-labelling supports it
)

In [15]:
DRONE_IPC_CODES = (
    "B64U",
)

In [16]:
DRONE_CPC_CODES = (
    "B64U",
    "B64C39/024",
)

In [17]:
def keyword_condition(terms):
    """OR-condition: any term appears in the title or in the abstract (case-insensitive)."""
    title = func.lower(TLS202_APPLN_TITLE.appln_title)
    abstract = func.lower(TLS203_APPLN_ABSTR.appln_abstract)
    return or_(
        *(title.like(f"%{t.lower()}%") for t in terms),
        *(abstract.like(f"%{t.lower()}%") for t in terms),
    )


def symbol_prefix_condition(symbol_column, codes):
    """OR-condition: the blank-stripped classification symbol starts with one of the codes."""
    symbol = func.replace(symbol_column, " ", "")
    return or_(*(symbol.like(code.replace(" ", "") + "%") for code in codes))


# Base columns shown for every application
APPLN_COLUMNS = (
    TLS201_APPLN.appln_id,
    TLS201_APPLN.appln_auth,
    TLS201_APPLN.appln_nr,
    TLS201_APPLN.appln_kind,
    TLS201_APPLN.appln_filing_date,
)


def query_by_keywords(*columns):
    """Applications whose title or abstract contains an Drone keyword."""
    return (
        db.query(*columns)
        .select_from(TLS201_APPLN)
        .outerjoin(TLS202_APPLN_TITLE, TLS201_APPLN.appln_id == TLS202_APPLN_TITLE.appln_id)
        .outerjoin(TLS203_APPLN_ABSTR, TLS201_APPLN.appln_id == TLS203_APPLN_ABSTR.appln_id)
        .filter(keyword_condition(DRONE_KEYWORDS))
    )


def query_by_ipc(*columns):
    """Applications with at least one IPC symbol in the Drone list."""
    return (
        db.query(*columns)
        .select_from(TLS201_APPLN)
        .join(TLS209_APPLN_IPC, TLS201_APPLN.appln_id == TLS209_APPLN_IPC.appln_id)
        .filter(symbol_prefix_condition(TLS209_APPLN_IPC.ipc_class_symbol, DRONE_IPC_CODES))
    )


def query_by_cpc(*columns):
    """Applications with at least one CPC symbol in the Drone list."""
    return (
        db.query(*columns)
        .select_from(TLS201_APPLN)
        .join(TLS224_APPLN_CPC, TLS201_APPLN.appln_id == TLS224_APPLN_CPC.appln_id)
        .filter(symbol_prefix_condition(TLS224_APPLN_CPC.cpc_class_symbol, DRONE_CPC_CODES))
    )

In [18]:
#KEYWORD SEARCH

q_keywords = query_by_keywords(
    *APPLN_COLUMNS,
    TLS202_APPLN_TITLE.appln_title,
    TLS203_APPLN_ABSTR.appln_abstract,
).distinct()

df_keywords = patstat.df(q_keywords)
df_keywords.head(10)

,appln_id,appln_auth,appln_nr,appln_kind,appln_filing_date,appln_title,appln_abstract
0,408663624,GB,201309002,A,2013-05-20,An unmanned aerial power plant drone,An unmanned aerial vehicle power plant drone i...
1,519747364,GR,20180100110,A,2018-03-08,FLOATING PLATFORM FOR MARITIME SURVEILLANCE AN...,The invention relates to an arrangement of one...
2,516768373,BR,102017025691,A,2017-11-29,plataforma para ensaios eólicos e uso,plataforma para ensaios eólicos e uso. a prese...
3,599390692,CY,221100094,T,2022-02-04,ΠΛΩΤΗ ΠΛΑΤΦΟΡΜΑ ΘΑΛΑΣΣΙΑΣ ΕΠΙΤΗΡΗΣΗΣ,"??a µ? epa?d??µ???, a?t???µ?, a?t?s??t????µe??..."
4,524539953,BR,102019014098,A,2019-07-08,Turbina de eixo vertical para aplicação eólica...,turbina de eixo vertical para aplicação eólica...
5,492764555,GB,201802746,A,2018-02-20,Unmanned aerial vehicles,"An unmanned aerial vehicle, UAV 100, is operab..."
6,524994221,DK,PA201870331,A,2018-06-07,APPARATUS AND METHOD FOR POSITIONING EQUIPMENT,A method for providing operational equipment a...
7,530889330,AU,2020202629,A,2020-04-17,Systems and methods for automatic vehicle imaging,Systems and methods are provided for automatic...
8,546391843,RU,2020125245,A,2019-12-04,LOW-TEMPERATURE SCANNING INFRARED ANALYZER OF ...,FIELD: measuring equipment.SUBSTANCE: inventio...
9,590349184,TW,111111954,A,2022-03-29,Maintenance method for structure,The present invention provides a maintenance m...


In [19]:
#IPC SEARCH
q_ipc = query_by_ipc(*APPLN_COLUMNS, TLS209_APPLN_IPC.ipc_class_symbol).distinct()

df_ipc = patstat.df(q_ipc)
df_ipc.head()

,appln_id,appln_auth,appln_nr,appln_kind,appln_filing_date,ipc_class_symbol
0,513740868,US,201916265314,A,2019-02-01,B64U 10/25
1,513740868,US,201916265314,A,2019-02-01,B64U 20/10
2,556168280,US,202117233087,A,2021-04-16,B64U 101/00
3,567233887,US,202117520459,A,2021-11-05,B64U 10/13
4,567233887,US,202117520459,A,2021-11-05,B64U 80/60


In [20]:
#CPC SEARCH
q_cpc = query_by_cpc(*APPLN_COLUMNS, TLS224_APPLN_CPC.cpc_class_symbol).distinct()

df_cpc = patstat.df(q_cpc)
df_cpc.head(10)

,appln_id,appln_auth,appln_nr,appln_kind,appln_filing_date,cpc_class_symbol
0,52104346,US,59725632,A,1932-03-07,B64U 10/60
1,54372193,US,96850378,A,1978-12-11,B64U 10/60
2,46566715,US,5881587,A,1987-06-05,B64U 70/30
3,50237261,US,36900589,A,1989-06-20,B64U 10/60
4,52709811,US,68000096,A,1996-07-15,B64C 39/024
5,52709811,US,68000096,A,1996-07-15,B64U 10/10
6,52709811,US,68000096,A,1996-07-15,B64U 10/25
7,52709811,US,68000096,A,1996-07-15,B64U 30/20
8,52709811,US,68000096,A,1996-07-15,B64U 30/21
9,52709811,US,68000096,A,1996-07-15,B64U 30/294


In [21]:
#COMBINE THE RESULTS
appln_id = TLS201_APPLN.appln_id.label("appln_id")

drone_ids = (
    query_by_keywords(appln_id)
    .union(query_by_ipc(appln_id), query_by_cpc(appln_id))
    .subquery()
)

q_drone = (
    db.query(*APPLN_COLUMNS, TLS202_APPLN_TITLE.appln_title)
    .select_from(TLS201_APPLN)
    .join(drone_ids, TLS201_APPLN.appln_id == drone_ids.c.appln_id)
    .outerjoin(TLS202_APPLN_TITLE, TLS201_APPLN.appln_id == TLS202_APPLN_TITLE.appln_id)
    .distinct()
    .order_by(TLS201_APPLN.appln_filing_date)
)

df_drone = patstat.df(q_drone)
df_drone.head(10)

,appln_id,appln_auth,appln_nr,appln_kind,appln_filing_date,appln_title
0,52104346,US,59725632,A,1932-03-07,Aerial signal device
1,50327951,US,3741504D,A,1971-05-06,CARGO HANDLING SYSTEM
2,54372193,US,96850378,A,1978-12-11,Wind driven apparatus for power generation
3,32854057,JP,15581581,A,1981-09-29,BRAKE DEVICE IN WINDMILL
4,46566715,US,5881587,A,1987-06-05,Point-landing method for non vertical take off...
5,50237261,US,36900589,A,1989-06-20,Remote controlled helicopter
6,42023759,NO,924962,A,1992-12-22,"System for boring, produksjon og andre operasj..."
7,26147721,JP,2213696,A,1996-01-12,AIR-CONDITIONING SYSTEM
8,52709811,US,68000096,A,1996-07-15,Landing and take-off assembly for vertical tak...
9,33253674,JP,16684698,A,1998-06-15,EXECUTION METHOD UNDER TRACK IN HOT SEASON


In [22]:
# How much does each strategy contribute? (counts of distinct applications)
pd.Series(
    {
        "Keywords": df_keywords["appln_id"].nunique(),
        "IPC": df_ipc["appln_id"].nunique(),
        "CPC": df_cpc["appln_id"].nunique(),
        "Total union": df_drone["appln_id"].nunique(),
    },
    name="Applications"
)

Keywords       1792
IPC             126
CPC            2190
Total union    3254
Name: Applications, dtype: int64